# 17 · Proper Scoring Rules & Probabilistic Evaluation

## What this notebook covers
Classifiers produce probabilities; regression models can produce full predictive distributions. Proper scoring rules evaluate the *quality of probabilistic forecasts* — rewarding forecasters who report their true beliefs.

## What makes a scoring rule proper?
A scoring rule S(p, y) is **proper** if the expected score is maximised (or minimised) when the forecaster reports their true belief p = P(y). This means:
- Proper scores incentivise honesty — you can't game them by reporting overconfident or underconfident probabilities
- Improper scores (like accuracy, MAPE) can be gamed

## The scoring rule family
| Rule | Type | Used for |
|------|------|---------|
| **Log-score (log-loss)** | Strictly proper, local | Classification, NLP perplexity |
| **Brier score** | Strictly proper, global | Binary probability forecasts |
| **CRPS** | Strictly proper, distributional | Regression with uncertainty, weather forecasting |
| **Energy score** | Multivariate CRPS generalisation | Vector-valued forecasts |

## Why CRPS is the modern standard for regression
CRPS (Continuous Ranked Probability Score) evaluates a full predictive CDF against a scalar observation. It reduces to MAE when the forecast is a point estimate, making it a proper generalisation of MAE to probabilistic forecasts. It is the dominant metric in numerical weather prediction and is increasingly used in financial and demand forecasting.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

print("Proper scoring rules: log-score, Brier, CRPS")

In [ ]:
# ── Classification: log-score and Brier ────────────────────────────────────────
X_c, y_c = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=0)
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.3, random_state=0)

clf = GradientBoostingClassifier(n_estimators=150, random_state=0)
clf.fit(X_tr_c, y_tr_c)
proba_c = clf.predict_proba(X_te_c)[:, 1]

ll  = log_loss(y_te_c, proba_c)
bs  = brier_score_loss(y_te_c, proba_c)

print(f"Log-loss (lower = better): {ll:.4f}")
print(f"Brier score (lower = better): {bs:.4f}")

# Skill score vs naive climatology baseline
base_proba = np.full_like(proba_c, y_te_c.mean())
bs_base = brier_score_loss(y_te_c, base_proba)
bss = 1 - bs / bs_base  # Brier Skill Score: 0 = no skill, 1 = perfect
print(f"Brier Skill Score (vs naive): {bss:.4f}  (>0 means better than guessing base rate)")

In [ ]:
# ── CRPS for regression ────────────────────────────────────────────────────────
X_r, y_r = make_regression(n_samples=1500, n_features=20, noise=30, random_state=0)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_r, y_r, test_size=0.3, random_state=0)
# Calibration split
X_tr2, X_cal, y_tr2, y_cal = train_test_split(X_tr_r, y_tr_r, test_size=0.2, random_state=0)

# Train quantile regressors for distributional forecast
quantiles = np.linspace(0.05, 0.95, 19)
q_models = {}
for q in quantiles:
    qr = QuantileRegressor(quantile=q, alpha=0.01, solver="highs")
    qr.fit(X_tr2, y_tr2)
    q_models[q] = qr

def crps_from_quantiles(y_obs, quantile_preds, quantile_levels):
    """
    Approximate CRPS from a set of quantile predictions using the formula:
    CRPS ≈ 2 * mean_q [ QL(y, F^{-1}(q), q) ]
    where QL is the quantile (pinball) loss.
    """
    crps_sum = 0.0
    for q, pred in zip(quantile_levels, quantile_preds.T):
        err = y_obs - pred
        crps_sum += np.mean(np.where(err >= 0, q * err, (q - 1) * err))
    return 2 * crps_sum / len(quantile_levels)

# Build quantile prediction matrix for test set
q_preds = np.column_stack([q_models[q].predict(X_te_r) for q in quantiles])
crps_val = crps_from_quantiles(y_te_r, q_preds, quantiles)

# Baseline CRPS: point forecast as delta distribution (equivalent to MAE)
reg_point = GradientBoostingRegressor(n_estimators=150, random_state=0)
reg_point.fit(X_tr2, y_tr2)
y_pred_point = reg_point.predict(X_te_r)
mae_baseline = np.mean(np.abs(y_te_r - y_pred_point))

print(f"CRPS (probabilistic forecast): {crps_val:.4f}")
print(f"MAE  (point forecast):         {mae_baseline:.4f}")
print(f"CRPS skill vs point: {1 - crps_val/mae_baseline:.4f}  (positive = probabilistic forecast adds value)")

In [ ]:
# Reliability: is our quantile forecast calibrated?
# For each quantile q, ~q*100% of observations should fall below the predicted quantile
coverage_by_q = []
for q, preds in zip(quantiles, q_preds.T):
    empirical_coverage = np.mean(y_te_r <= preds)
    coverage_by_q.append({"quantile": q, "nominal": q, "empirical": empirical_coverage})

coverage_df = pd.DataFrame(coverage_by_q)
print(coverage_df[["quantile","nominal","empirical"]].round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Log-score decomposition illustration (Brier score reliability diagram)
from sklearn.calibration import calibration_curve
fop, mpv = calibration_curve(y_te_c, proba_c, n_bins=10)
axes[0].plot([0,1],[0,1],"k--",lw=1,label="Perfect calibration")
axes[0].plot(mpv, fop, "o-", color="steelblue", label=f"Model (BS={bs:.3f})")
axes[0].fill_between(mpv, fop, mpv, alpha=0.2, color="tomato", label="Calibration error")
axes[0].set_title("Reliability diagram (Brier score)")
axes[0].set_xlabel("Mean predicted probability"); axes[0].set_ylabel("Fraction positive")
axes[0].legend(fontsize=8)

# Quantile calibration (PIT diagram)
axes[1].plot(coverage_df["nominal"], coverage_df["empirical"], "o-", color="steelblue", label="Empirical")
axes[1].plot([0,1],[0,1],"k--",lw=1,label="Perfect calibration")
axes[1].set_xlabel("Nominal quantile"); axes[1].set_ylabel("Empirical coverage")
axes[1].set_title("Quantile calibration (PIT diagram)")
axes[1].legend(fontsize=8)

# Example predictive distribution vs observation for one test point
idx = 50
q_pred_example = q_preds[idx]
obs = y_te_r[idx]
axes[2].plot(quantiles, q_pred_example, "o-", color="steelblue", lw=1.5, markersize=4, label="Predicted quantiles")
axes[2].axvline(np.mean(y_te_r <= q_pred_example[-1]), color="gray", lw=0.5)
axes[2].axhline(obs, color="red", linestyle="--", lw=1.5, label=f"Observed={obs:.1f}")
axes[2].set_xlabel("Quantile level"); axes[2].set_ylabel("Value")
axes[2].set_title("Predictive distribution (single test point)")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/17_proper_scoring_rules.png", dpi=120)
plt.show()

## Summary
| Scenario | Recommended metric |
|----------|-------------------|
| Binary classification, calibration matters | Brier score + Brier Skill Score |
| Classification, sharp discrimination focus | Log-loss |
| Regression, point forecast | MAE / RMSE depending on loss preference |
| Regression, probabilistic forecast | CRPS |
| Multi-step time-series forecast | CRPS per horizon, aggregated |

CRPS is the principled way to evaluate any regression model that produces uncertainty estimates — including Bayesian models, conformal intervals (NB 15), and quantile regressors. If your model outputs a distribution, evaluate it with CRPS, not just the median MAE.